In [1]:
import os, sys
sys.path.append('../src')
from FIT_python.data_split_and_summary.datasplit_and_summary_wraper import prepare_all_splits

# Daten-Splits sind bereits vorhanden, Aufruf bei Bedarf:
#prepare_all_splits()

In [2]:
from FIT_python.plot_style import apply_styleapply_style()import pandas as pdfrom pathlib import Pathfrom FIT_python.pipeline_individual_id.data_split import sequential_holdout_idsfrom FIT_python.pipeline_individual_id.pair_generation import build_pairwise_comparisonsfrom FIT_python.pipeline_individual_id.data_split import assign_foldsfrom FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaselinefrom FIT_python.pipeline_individual_id.supervised_umap_pipeline import run as run_umapfrom FIT_python.pipeline_individual_id.siamese_pipeline import run as run_siamesefrom FIT_python.pipeline_individual_id.evaluation import compute_confusionimport seaborn as sns, matplotlib.pyplot as pltfrom FIT_python.data_split_and_summary.data_import_utils import (    load_and_prep_df_individual,    get_feature_cols,)from FIT_python.config import RESULTS_DATA_DIR, SPLITS_DIR

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [4]:
splits_root = Path("data/splits/eurasian_otter")models_root = (    RESULTS_DATA_DIR    / "eurasian_otter_random_search_standard_metrics"    / "best_maj_test_pct")output_root = Path("results/data/pairwise_comp_eurasian_otter")output_root.mkdir(parents=True, exist_ok=True)# 6) Loop über Speziesspecies_dir = Path(SPLITS_DIR) / "eurasian_otter"species = "eurasian_otter"print(f"\n🔄 Processing species: {species}")# 6.1) Train/Test laden & Fold entfernendf = load_and_prep_df_individual(    species, Path(SPLITS_DIR))  # falls hier bugs valid vales f und m checkenfeature_cols = [    c for c in df.columns if c.startswith(("dist", "ang", "t", "v")) and c != "trail"]ids = df["individual_id"].unique()split = sequential_holdout_ids(ids, val_sizes=[4, 8, 12,16,20], n_iter=5, random_state=0)[0]train_df = df[df["individual_id"].isin(split["train_ids"])]val_df = df[df["individual_id"].isin(split["val_ids"])]val_df["_group_id"] = val_df["individual_id"].where(val_df["individual_id"].notna() & (val_df["individual_id"] != "unknown"), val_df["trail"]).astype(str)sex_map = val_df.groupby("_group_id")['sex'].first().fillna('unknown').replace('', 'unknown').to_dict()fallback_map = val_df.groupby("_group_id")['individual_id'].first().apply(lambda x: pd.isna(x) or x=='unknown').to_dict()val_comps, _ = build_pairwise_comparisons(val_df, id_col="individual_id", sex_map=sex_map, fallback_map=fallback_map, trail_col="trail")val_comps = assign_folds(val_comps, sex_map, n_folds=3, random_state=42)# In ein DataFrame überführendf_val_comps = pd.DataFrame(val_comps)


🔄 Processing species: eurasian_otter


size_a:   0%|          | 0/4 [00:00<?, ?it/s]
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\joblib\_store_backends.py:219: CacheWarning: Unable to cache to disk. Possibly a race condition in the creation of the directory. Exception: [Errno 2] No such file or directory: 'C:\\Users\\Frede\\OneDrive\\Projects and Disschaptors\\FIT_package\\src\\FIT_python\\pipeline_individual_id\\__cache__\\FIT_python\\pipeline_individual_id\\geometric_pairwise_projection\\generate_pairwise_comparisons_from_df\\374c7217d3cdaafd36f9286b9fb71058\\output.pkl.thread-2697587205200-pid-11988'.
  warnings.warn(


In [5]:
val_comps

[{'ind_a': 'japet',
  'ind_b': 'leni',
  'samples_a': [2500, 2504],
  'samples_b': [2602, 2610],
  'trail_a_id': 'japet_2a',
  'trail_b_id': 'leni_2a',
  'same_individual': False,
  'fold': 0},
 {'ind_a': 'japet',
  'ind_b': 'leni',
  'samples_a': [2518, 2517],
  'samples_b': [2604, 2613],
  'trail_a_id': 'japet_2b',
  'trail_b_id': 'leni_2b',
  'same_individual': False,
  'fold': 0},
 {'ind_a': 'japet',
  'ind_b': 'leni',
  'samples_a': [2511, 2507],
  'samples_b': [2604, 2603],
  'trail_a_id': 'japet_2c',
  'trail_b_id': 'leni_2c',
  'same_individual': False,
  'fold': 0},
 {'ind_a': 'japet',
  'ind_b': 'leni',
  'samples_a': [2517, 2511],
  'samples_b': [2610, 2599],
  'trail_a_id': 'japet_2d',
  'trail_b_id': 'leni_2d',
  'same_individual': False,
  'fold': 0},
 {'ind_a': 'japet',
  'ind_b': 'paulinchen',
  'samples_a': [2517, 2514],
  'samples_b': [2667, 2671],
  'trail_a_id': 'japet_2e',
  'trail_b_id': 'paulinchen_2a',
  'same_individual': False,
  'fold': 0},
 {'ind_a': 'japet'

In [ ]:
import pandas as pdimport seaborn as snsimport matplotlib.pyplot as pltfrom sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import StratifiedKFold, cross_val_predictfrom FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaselinefrom FIT_python.pipeline_individual_id.evaluation import compute_confusionsplits = sequential_holdout_ids(ids, val_sizes=[3, 5, 9], n_iter=10, random_state=0)cms_base, cms_umap, cms_siam = [], [], []for split in splits:    train_df = df[df["individual_id"].isin(split["train_ids"])]    val_df = df[df["individual_id"].isin(split["val_ids"])]    val_df['_group_id'] = val_df['individual_id'].where(val_df['individual_id'].notna() & (val_df['individual_id'] != 'unknown'), val_df['trail']).astype(str)    sex_map = val_df.groupby('_group_id')['sex'].first().fillna('unknown').replace('', 'unknown').to_dict()    fallback_map = val_df.groupby('_group_id')['individual_id'].first().apply(lambda x: pd.isna(x) or x=='unknown').to_dict()    val_comps, _ = build_pairwise_comparisons(val_df, id_col='individual_id', sex_map=sex_map, fallback_map=fallback_map, trail_col='trail')    val_comps = assign_folds(val_comps, sex_map, n_folds=2, random_state=42)    res_base = DistanceBaseline.run(train_df, val_comps, feature_cols, val_df=val_df)    Xb = res_base.filter(regex="^dist_")    yb = res_base["same_individual"].map({"True": 1, "False": 0})    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)    lr = LogisticRegression(max_iter=200)    pred_b = cross_val_predict(lr, Xb, yb, cv=cv)    cm_b = compute_confusion(pd.DataFrame({"same_individual": yb, "pred": pred_b}))    cms_base.append(cm_b)    res_umap, _ = run_umap(train_df, val_comps, feature_cols, val_df=val_df)    res_umap = res_umap[res_umap["same_individual"].isin([True, False, 'True', 'False', 1, 0])]    res_umap["pred"] = (res_umap["pred_same_proba"] > 0.5).astype(int)    cm_u = compute_confusion(res_umap, true_col="same_individual", pred_col="pred")    cms_umap.append(cm_u)    res_siam = run_siamese(train_df, val_comps, feature_cols, val_df=val_df)    df_siam = pd.DataFrame({        'same_individual': [c['same_individual'] for c in val_comps],        'pred_same': [r['pred_same'] for r in res_siam],    })    df_siam = df_siam[df_siam['same_individual'].isin([True, False, 'True', 'False', 1, 0])]    df_siam['pred'] = (df_siam['pred_same'] > 0.5).astype(int)    cm_s = compute_confusion(df_siam, true_col='same_individual', pred_col='pred')    cms_siam.append(cm_s)def sum_cms(cms):    total = cms[0].copy()    for cm in cms[1:]:        total += cm    return totalcm_base_total = sum_cms(cms_base)cm_umap_total = sum_cms(cms_umap)cm_siam_total = sum_cms(cms_siam)display(cm_base_total, cm_umap_total, cm_siam_total)

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\joblib\_store_backends.py:219: CacheWarning: Unable to cache to disk. Possibly a race condition in the creation of the directory. Exception: [Errno 2] No such file or directory: 'C:\\Users\\Frede\\OneDrive\\Projects and Disschaptors\\FIT_package\\src\\FIT_python\\pipeline_individual_id\\__cache__\\FIT_python\\pipeline_individual_id\\geometric_pairwise_projection\\generate_pairwise_comparisons_from_df\\2c8b34b4a8afd5fc6c5fccd0105c7a40\\output.pkl.thread-2697587205200-pid-11988'.
  warnings.warn(
Processing Pairs:   0%|          | 0/96 [00:00<?, ?it/s]

  0%|          | 0/96 [00:00<?, ?it/s]

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Processing Pairs:   0%|          | 0/96 [00:24<?, ?it/s]
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
sns.heatmap(cm_base_total, annot=True, fmt='d', ax=axes[0])
axes[0].set_title('Baseline')
sns.heatmap(cm_umap_total, annot=True, fmt='d', ax=axes[1])
axes[1].set_title('Supervised UMAP')
sns.heatmap(cm_siam_total, annot=True, fmt='d', ax=axes[2])
axes[2].set_title('Siamese')
fig.tight_layout()
plt.show()

In [ ]:
# Load the test set and prepare trails without generating new windows
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

test_df = pd.read_parquet(species_dir / "test.parquet")
test_df = test_df.drop(columns=["Fold"], errors="ignore")

trails_per_animal = {}
for ind, grp_ind in test_df.groupby("individual_id"):
    trails_per_animal[ind] = {}
    for idx, (trail_id, g_trail) in enumerate(grp_ind.groupby("trail")):
        ids_list = g_trail["id"].astype(str).tolist()
        size = len(ids_list)
        trails_per_animal[ind].setdefault(size, []).append((idx, ids_list))

In [ ]:
test_df["_group_id"] = test_df["individual_id"].where(test_df["individual_id"].notna() & (test_df["individual_id"] != "unknown"), test_df["trail"]).astype(str)sex_map = test_df.groupby("_group_id")["sex"].first().fillna('unknown').replace('', 'unknown').to_dict()fallback_map = test_df.groupby("_group_id")["individual_id"].first().apply(lambda x: pd.isna(x) or x=='unknown').to_dict()test_comps, _ = build_pairwise_comparisons(test_df, id_col="individual_id", sex_map=sex_map, fallback_map=fallback_map, trail_col="trail")test_comps = assign_folds(test_comps, sex_map, n_folds=1, random_state=42)df_test_comps = pd.DataFrame(test_comps)

In [ ]:
# Run inference on the test set
res_base_test = DistanceBaseline.run(train_df, test_comps, feature_cols, val_df=test_df)
res_umap_test, _ = run_umap(train_df, test_comps, feature_cols, val_df=test_df)
res_siam_test = run_siamese(train_df, test_comps, feature_cols, val_df=test_df)

In [ ]:
# Helper function to plot hierarchical clustering with Ward linkage

def plot_clustering(res_df, method_name):
    dists = res_df[["trail_a_id", "trail_b_id", "dist_euclidean"]]
    trails = sorted(set(dists["trail_a_id"]).union(dists["trail_b_id"]))
    index = {t: i for i, t in enumerate(trails)}
    mat = np.zeros((len(trails), len(trails)))
    for _, row in dists.iterrows():
        i = index[row.trail_a_id]
        j = index[row.trail_b_id]
        mat[i, j] = row.dist_euclidean
        mat[j, i] = row.dist_euclidean

    Z = linkage(squareform(mat), method="ward")

    trail_to_ind = {c["trail_a_id"]: c["ind_a"] for c in test_comps}
    trail_to_ind.update({c["trail_b_id"]: c["ind_b"] for c in test_comps})
    inds = sorted(set(trail_to_ind[t] for t in trails))
    palette = sns.color_palette("husl", len(inds))
    color_map = dict(zip(inds, palette))

    dendrogram(Z, labels=trails, orientation="top", above_threshold_color="grey")
    ax = plt.gca()
    for lbl in ax.get_xmajorticklabels():
        ind = trail_to_ind.get(lbl.get_text())
        if ind is not None:
            lbl.set_color(color_map[ind])
    ax.set_ylabel("distance")
    plt.title(f"{method_name} clustering")
    plt.show()

In [ ]:
# Plot clusters for all methods
plot_clustering(res_base_test, "Baseline")
plot_clustering(res_umap_test, "Supervised UMAP")
res_siam_df = pd.DataFrame({
    "trail_a_id": [c["trail_a_id"] for c in test_comps],
    "trail_b_id": [c["trail_b_id"] for c in test_comps],
    "dist_euclidean": [1 - r["pred_same"] for r in res_siam_test],
})
plot_clustering(res_siam_df, "Siamese")